# 📘 Logistic Regression — Tuning Cheat Sheet

### 1. Important Variables

| Variable | Purpose |
|---|---|
| `X` | Input features |
| `y` | Target/output labels |
| `test_size=0.25` | Uses 25% data for testing |
| `random_state=42` | Ensures reproducible splitting |
| `stratify=y` | Maintains class proportions |
| `y_pred` | Stores predicted labels |

### 2. Important Hyperparameters

| Hyperparameter | Why Use It? |
|---|---|
| `C` | Controls regularization; smaller value means stronger regularization |
| `penalty` | Regularization type (`l1`, `l2`) |
| `solver` | Algorithm used to train the model |
| `max_iter` | Maximum training iterations |
| `class_weight` | Helps handle imbalanced classes |

### 3. Preprocessing & Tuning

| Variable / Word | Purpose |
|---|---|
| `StandardScaler()` | Scales features |
| `Pipeline()` | Combines scaling and model training |
| `param_grid` | Stores hyperparameter values to test |
| `GridSearchCV()` | Finds good parameters using cross-validation |
| `cv=4` | Uses 4 cross-validation folds |
| `scoring="accuracy"` | Uses accuracy to compare models |
| `n_jobs=-1` | Uses available CPU cores for parallel processing |
| `best_params_` | Best hyperparameter combination |
| `best_score_` | Best mean cross-validation score |
| `best_estimator_` | Best fitted model/pipeline |

### 4. Evaluation Terms

| Term | Meaning |
|---|---|
| Accuracy | Overall proportion of correct predictions |
| Precision | Correct positive predictions among predicted positives |
| Recall | Correct positive predictions among actual positives |
| F1-score | Balance between precision and recall |
| Overfitting | Model performs well on training data but poorly on unseen data |
| Data leakage | Unavailable or inappropriate information influences training |
| Cross-validation | Evaluates model performance across multiple data folds |

### 5. Essential Workflow

**Prepare `X` and `y` → Split Data → Scale Features → Train Model → Tune with `GridSearchCV` → Evaluate on Test Data**

**Remember:** Fit preprocessing and tune hyperparameters using training data only. Use the test set for final evaluation.

In [ ]:
# Cell 1: Import required librari

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Cell 2: Create the fruit dat

data = {
    "Weight": [
        150, 160, 170, 140, 155, 165, 175, 145,
        130, 135, 125, 138, 145, 128, 132, 140,
        180, 190, 185, 195, 120, 118, 122, 115
    ],
    "Sweetness": [
        7, 8, 7, 6, 8, 7, 9, 6,
        4, 5, 3, 4, 5, 3, 4, 5,
        8, 9, 8, 9, 3, 2, 3, 2
    ],
    "Fruit": [
        "Apple", "Apple", "Apple", "Apple",
        "Apple", "Apple", "Apple", "Apple",
        "Orange", "Orange", "Orange", "Orange",
        "Orange", "Orange", "Orange", "Orange",
        "Apple", "Apple", "Apple", "Apple",
        "Orange", "Orange", "Orange", "Orange"
    ]
}

df = pd.DataFrame(data)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nClass counts:\n", df["Fruit"].value_counts())

   Weight  Sweetness  Fruit
0     150          7  Apple
1     160          8  Apple
2     170          7  Apple
3     140          6  Apple
4     155          8  Apple

Dataset shape: (24, 3)

Class counts:
 Fruit
Apple     12
Orange    12
Name: count, dtype: int64


In [3]:
# Cell 3: Separate X and y

X = df[["Weight", "Sweetness"]]
y = df["Fruit"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (24, 2)
Target shape: (24,)


In [4]:
# Cell 4: Split training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 18
Testing samples: 6


In [5]:
# Cell 5: Train baseline Logistic Regression

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.6666666666666666


In [6]:
# Cell 6: Compare different C values

for c_value in [0.01, 0.1, 1, 10, 100]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=c_value, max_iter=2000))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    score = accuracy_score(y_test, predictions)

    print(f"C={c_value}: Accuracy={score:.2f}")

C=0.01: Accuracy=0.67
C=0.1: Accuracy=0.67
C=1: Accuracy=0.67
C=10: Accuracy=0.67
C=100: Accuracy=0.67


In [7]:
# Cell 7: Compare regularization penalties

for penalty_value in ["l1", "l2"]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            penalty=penalty_value,
            solver="saga",
            C=1,
            max_iter=5000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    score = accuracy_score(y_test, predictions)

    print(f"Penalty={penalty_value}: Accuracy={score:.2f}")

Penalty=l1: Accuracy=1.00
Penalty=l2: Accuracy=0.67


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

In [8]:
# Cell 8: Compare compatible solvers

for solver_value in ["lbfgs", "liblinear", "saga"]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            solver=solver_value,
            penalty="l2",
            max_iter=5000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    score = accuracy_score(y_test, predictions)

    print(f"Solver={solver_value}: Accuracy={score:.2f}")

Solver=lbfgs: Accuracy=0.67
Solver=liblinear: Accuracy=0.67
Solver=saga: Accuracy=0.67


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [9]:
# Cell 9: Compare maximum iterations

for iterations in [100, 500, 1000, 3000]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=iterations,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    score = accuracy_score(y_test, predictions)

    print(f"max_iter={iterations}: Accuracy={score:.2f}")

max_iter=100: Accuracy=0.67
max_iter=500: Accuracy=0.67
max_iter=1000: Accuracy=0.67
max_iter=3000: Accuracy=0.67


In [10]:
# Cell 10: Compare class weights

for weight_value in [None, "balanced"]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            class_weight=weight_value,
            max_iter=2000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    score = accuracy_score(y_test, predictions)

    print(f"class_weight={weight_value}: Accuracy={score:.2f}")

class_weight=None: Accuracy=0.67
class_weight=balanced: Accuracy=0.67


In [11]:
# Cell 11: Define the Logistic Regression search space

param_grid = [
    {
        "lr__C": [0.01, 0.1, 1, 10, 100],
        "lr__penalty": ["l2"],
        "lr__solver": ["lbfgs", "liblinear"],
        "lr__class_weight": [None, "balanced"]
    },
    {
        "lr__C": [0.01, 0.1, 1, 10, 100],
        "lr__penalty": ["l1", "l2"],
        "lr__solver": ["saga"],
        "lr__class_weight": [None, "balanced"]
    }
]

In [12]:
# Cell 12: Find the best hyperparameters

grid_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=3000,
            random_state=42
        ))
    ]),
    param_grid=param_grid,
    cv=4,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid_search.best_score_)

Best Parameters:
{'lr__C': 0.01, 'lr__class_weight': None, 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}

Best Cross-Validation Accuracy:
1.0


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [13]:
# Cell 13: Evaluate on unseen test data

best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(
    y_test,
    tuned_predictions
)

print("Tuned Test Accuracy:", tuned_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    tuned_predictions,
    zero_division=0
))

Tuned Test Accuracy: 0.6666666666666666

Classification Report:
              precision    recall  f1-score   support

       Apple       1.00      0.33      0.50         3
      Orange       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6



In [14]:
# Cell 14: Compare test accuracy

print(f"Baseline Accuracy: {baseline_accuracy:.2%}")
print(f"Tuned Accuracy:    {tuned_accuracy:.2%}")

difference = tuned_accuracy - baseline_accuracy

print(
    f"Accuracy Difference: "
    f"{difference * 100:+.2f} percentage points"
)

Baseline Accuracy: 66.67%
Tuned Accuracy:    66.67%
Accuracy Difference: +0.00 percentage points


In [15]:
# Cell 15: Predict the class of a new fruit

new_fruit = pd.DataFrame({
    "Weight": [155],
    "Sweetness": [7]
})

prediction = best_model.predict(new_fruit)

print("Predicted fruit:", prediction[0])

Predicted fruit: Apple
